# LightOnOCR: HF → S3/MinIO

Download the model from HF

In [ ]:
%pip install -q boto3 huggingface_hub
import os

for k in ["AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "AWS_S3_ENDPOINT", "AWS_S3_BUCKET", "HF_TOKEN"]:
    v = os.environ.get(k, "")
    ok = bool(v) and v not in ("<YOUR_HF_TOKEN>",)
    show = f"{v[:4]}…" if (v and ("SECRET" in k or k == "HF_TOKEN")) else (v or "(missing)")
    print(f"{'OK' if ok else '!!'}  {k}: {show}")

In [ ]:
import os
import boto3
import urllib3
from huggingface_hub import snapshot_download

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)


def _env(name: str) -> str:
    v = os.environ.get(name, "").strip()
    if not v:
        raise SystemExit(f"Missing {name}. Mount minio + huggingface-token (envFrom).")
    return v


hf_token = _env("HF_TOKEN")
if hf_token in ("<YOUR_HF_TOKEN>", "changeme"):
    raise SystemExit("Set HF_TOKEN.")

aws_key = _env("AWS_ACCESS_KEY_ID")
aws_secret = _env("AWS_SECRET_ACCESS_KEY")
endpoint = _env("AWS_S3_ENDPOINT")
bucket_name = (os.environ.get("AWS_S3_BUCKET") or "models").strip()
verify_ssl = os.environ.get("S3_VERIFY_SSL", "false").lower() in ("1", "true", "yes")
os.environ.setdefault("AWS_DEFAULT_REGION", "us-east-1")

model_id = os.environ.get("MODEL_ID", "lightonai/LightOnOCR-2-1B")
s3_folder = os.environ.get("S3_FOLDER", "lighton-ocr")

print(f"--- Downloading {model_id} ---")
local_path = snapshot_download(
    repo_id=model_id,
    token=hf_token,
    # Include chat_template.jinja; required for vLLM chat (transformers 4.44+).
    allow_patterns=["*.json", "*.bin", "*.safetensors", "*.py", "*.txt", "*.jinja", "*.jinja2"],
    ignore_patterns=["*.msgpack", "*.h5"],
)

s3 = boto3.client(
    "s3",
    endpoint_url=endpoint,
    aws_access_key_id=aws_key,
    aws_secret_access_key=aws_secret,
    verify=verify_ssl,
)
try:
    s3.create_bucket(Bucket=bucket_name)
    print(f"Created bucket: {bucket_name}")
except Exception:
    pass

print(f"--- Uploading to {bucket_name}/{s3_folder} ---")
for root, _, files in os.walk(local_path):
    for file in files:
        full_path = os.path.join(root, file)
        rel_path = os.path.relpath(full_path, local_path)
        s3_key = f"{s3_folder}/{rel_path}"
        print(f"Uploading {rel_path}...")
        s3.upload_file(full_path, bucket_name, s3_key)

print("\nDONE:", f"s3://{bucket_name}/{s3_folder}/")

In [ ]:
# Check contents of bucket
import os
import boto3

# From env (same names as the MinIO / OpenShift connection secret)
s3 = boto3.client(
    "s3",
    # endpoint_url=os.environ.get("AWS_S3_ENDPOINT"),  # e.g. http://minio-service.minio.svc:9000
    endpoint_url="http://rook-ceph-rgw-ocs-storagecluster-cephobjectstore.openshift-storage.svc:80",
    aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
    aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
    region_name=os.environ.get("AWS_DEFAULT_REGION", "us-east-1"),
)

bucket = os.environ.get("AWS_S3_BUCKET", "models")
prefix = ""  # optional, or "" for the whole bucket

paginator = s3.get_paginator("list_objects_v2")
for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
    for obj in page.get("Contents", []):
        print(f"{obj['Key']}\t{obj['Size']}")